# Nemotron Reasoning Challenge — LoRA Training

**Platform**: Google Colab (A100 High-RAM 80GB)
**Model**: Downloaded via `kagglehub` — no Drive upload needed

**Workflow:**
1. Setup — install deps, clone repo, configure kagglehub auth
2. Data — upload train.csv, generate formatted JSONL
3. Model — download via kagglehub, mount Drive for output persistence
4. Baseline — ~500 problems to estimate Nano accuracy per category
5. Train LoRA — SFT with bf16 (or QLoRA 4-bit fallback)
6. Eval — compare baseline vs LoRA on val set
7. Export — save adapter to Drive, download zip for Kaggle submission

**Runtime setup**: Runtime → Change runtime type → **A100** + **High RAM**

**Before running**: Have `data/train.csv` ready to upload (competition data).

**Kaggle API**: Add your Kaggle API token as a Colab Secret named `KAGGLE_API_TOKEN`
(one-time setup: Colab sidebar → Secrets → + New secret → paste token from kaggle.json)

In [ ]:
# -- Cell 1: Setup -------------------------------------------------
import subprocess, sys
from pathlib import Path

def _run(cmd, label):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'=== {label} FAILED ===')
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f'{label} failed')
    print(f'{label}: OK')

# Training dependencies
_run([sys.executable, '-m', 'pip', 'install',
    'peft', 'transformers', 'trl', 'datasets',
    'accelerate', 'bitsandbytes', 'kagglehub',
], 'training deps')

# vLLM (for eval only — separate install to isolate failures)
_run([sys.executable, '-m', 'pip', 'install', 'vllm'], 'vllm')

# Clone repo (pipeline code + cipher CoT data)
if not Path('/content/repo/nemotron-challenge/pipeline/utils.py').exists():
    _run(['git', 'clone', '--depth=1',
          'https://github.com/Zeno-MS/Kaggle-Nvidia.git',
          '/content/repo'], 'git clone')

# Locate pipeline
def _find_repo_root():
    for p in [Path('/content/repo/nemotron-challenge'), Path.cwd()]:
        if (p / 'pipeline' / 'utils.py').is_file():
            return p
    return None

repo_root = _find_repo_root()
if repo_root is None:
    raise FileNotFoundError('Pipeline not found. Check git clone output above.')

REPO = str(repo_root)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from pipeline.utils import load_problems, verify, extract_final_answer
from pipeline.evaluator import Evaluator, stratified_sample
print(f'Setup complete. Pipeline loaded from {REPO}')

In [ ]:
# -- Cell 2: Upload data + generate formatted JSONL ----------------
import os, subprocess

from google.colab import files

print("Upload train.csv from your local machine:")
uploaded = files.upload()

data_dir = f"{REPO}/data"
os.makedirs(data_dir, exist_ok=True)
for name, content in uploaded.items():
    dest = f"{data_dir}/{name}"
    with open(dest, "wb") as f:
        f.write(content)
    print(f"Saved: {dest}")

# Generate base formatted JSONL from train.csv
print("\nGenerating base training data...")
result = subprocess.run(
    [sys.executable, "pipeline/formatter.py"],
    cwd=REPO, capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": REPO},
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Formatter failed")

# Merge GPT-4.1 cipher CoTs (included in repo via git clone)
print("\nMerging GPT-4.1 cipher CoTs...")
result2 = subprocess.run(
    [sys.executable, "scripts/generate_cipher_cot.py", "--merge-only"],
    cwd=REPO, capture_output=True, text=True,
    env={**os.environ, "PYTHONPATH": REPO},
)
print(result2.stdout[-1000:])
if result2.returncode != 0:
    print("STDERR:", result2.stderr[-1000:])
    raise RuntimeError("Cipher merge failed")

TRAIN_JSONL = f"{data_dir}/train_formatted.jsonl"
VAL_JSONL   = f"{data_dir}/val_formatted.jsonl"
TRAIN_CSV   = f"{data_dir}/train.csv"
print(f"\nData ready: {TRAIN_JSONL}")

In [ ]:
# -- Cell 3: Download model + mount Drive ----------------------------
import kagglehub
import os

# Download model via kagglehub (uses Colab Secret KAGGLE_API_TOKEN)
# First run downloads ~60GB; cached for the rest of the session
print("Downloading model via kagglehub (may take 5-10 min on first run)...")
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Model ready: {MODEL_PATH}")

# Mount Drive for output persistence across sessions
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUT   = "/content/drive/MyDrive/Nemotron"
ADAPTER_DIR = "/content/lora_adapter"
RESULTS_DIR = "/content/results"

os.makedirs(DRIVE_OUT,   exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Outputs → {DRIVE_OUT}")

In [ ]:
# -- Cell 3b: Config -------------------------------------------------
# LoRA hyperparameters (match competition demo exactly)
LORA_RANK    = 32
LORA_ALPHA   = 16
LORA_TARGETS = r".*\.(in_proj|out_proj|up_proj|down_proj)$"
LORA_DROPOUT = 0.05

# Training hyperparameters
NUM_EPOCHS    = 3
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 4096

# Quantization toggle
# False = bf16 (needs A100 80GB High-RAM, matches competition eval exactly)
# True  = QLoRA 4-bit (fits A100 40GB, slight quality tradeoff)
USE_QLORA = False

print(f'LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}')
print(f'Training: lr={LEARNING_RATE}, epochs={NUM_EPOCHS}, batch={BATCH_SIZE}x{GRAD_ACCUM}')
print(f'Quantization: {"QLoRA 4-bit" if USE_QLORA else "bf16 (High-RAM required)"}')

## Baseline Eval

Run unmodified Nano on ~500 stratified problems to estimate per-category accuracy.
~15 min on A100. Tells us where to focus LoRA training.

In [ ]:
# ── Load problems and sample ───────────────────────────────────────
problems = load_problems(TRAIN_CSV)
sample = stratified_sample(problems, n_per_category=83)  # ~500 total
print(f"Sample: {len(sample)} problems")

In [ ]:
# ── Run baseline inference ─────────────────────────────────────────
baseline_ev = Evaluator(model_path=MODEL_PATH)  # No LoRA
baseline_summary = baseline_ev.run(sample)
Evaluator.report(baseline_summary)
baseline_ev.save_results(baseline_summary, f"{RESULTS_DIR}/baseline_results.jsonl")

In [ ]:
# ── Save baseline report to Drive ─────────────────────────────────
import json
with open(f"{DRIVE_OUT}/baseline_summary.json", "w") as f:
    json.dump({
        "accuracy": baseline_summary.accuracy,
        "total": baseline_summary.total,
        "correct": baseline_summary.correct,
        "per_category": baseline_summary.per_category,
    }, f, indent=2)
print("Baseline saved to Drive.")

# Clean up model from GPU memory before training
del baseline_ev._llm
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

## Train LoRA

SFT on formatted training data using PEFT + TRL SFTTrainer.

In [ ]:
# ── Load formatted training data ───────────────────────────────────
from datasets import load_dataset

train_ds = load_dataset("json", data_files=TRAIN_JSONL, split="train")
val_ds   = load_dataset("json", data_files=VAL_JSONL,   split="train")

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print("Sample prompt (first 200 chars):")
print(train_ds[0]["prompt"][:200])
print("Sample response (first 200 chars):")
print(train_ds[0]["response"][:200])

In [ ]:
# -- Load model and apply LoRA ---------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Model loading — bf16 or QLoRA depending on config
if USE_QLORA:
    print("Loading model in 4-bit (QLoRA)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    print("Loading model in bf16 (requires A100 80GB High-RAM)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA applied.")

In [ ]:
# ── Format data for SFTTrainer ─────────────────────────────────────
def format_for_sft(example):
    """Combine prompt + response into a single text field for SFT."""
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_ds.map(format_for_sft)
val_ds   = val_ds.map(format_for_sft)
print("Data formatted for SFT.")
print("Sample text (first 300 chars):")
print(train_ds[0]["text"][:300])

In [ ]:
# -- Train ----------------------------------------------------------
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
# -- Save LoRA adapter -----------------------------------------------
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Copy to Drive for persistence across sessions
import shutil
shutil.copytree(ADAPTER_DIR, f"{DRIVE_OUT}/lora_adapter_v1", dirs_exist_ok=True)
print(f"Adapter saved to Drive: {DRIVE_OUT}/lora_adapter_v1")

# Free training resources for eval
del trainer, model
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

## Evaluate

Run the trained LoRA on the validation split and compare to baseline.

In [ ]:
# ── Load val problems ──────────────────────────────────────────────
import json
val_ids = set()
with open(VAL_JSONL) as f:
    for line in f:
        val_ids.add(json.loads(line)["id"])

all_problems = load_problems(TRAIN_CSV)
val_problems = [p for p in all_problems if p.id in val_ids]
print(f"Val set: {len(val_problems)} problems")

In [ ]:
# ── Evaluate with LoRA ─────────────────────────────────────────────
lora_ev = Evaluator(model_path=MODEL_PATH, lora_path=ADAPTER_DIR)
lora_summary = lora_ev.run(val_problems)
Evaluator.report(lora_summary)
lora_ev.save_results(lora_summary, f"{RESULTS_DIR}/lora_v1_results.jsonl")

In [ ]:
# ── Compare baseline vs LoRA ───────────────────────────────────────
print("\n=== Baseline vs LoRA Comparison ===")
print(f"{'Category':<25} {'Baseline':>10} {'LoRA v1':>10} {'Delta':>8}")
print("-" * 57)

base_cats = baseline_summary.per_category
lora_cats = lora_summary.per_category

for cat in sorted(lora_cats):
    base_acc = base_cats.get(cat, {}).get("accuracy", 0.0)
    lora_acc = lora_cats[cat]["accuracy"]
    delta = lora_acc - base_acc
    sign = "+" if delta >= 0 else ""
    print(f"{cat:<25} {base_acc*100:>9.1f}% {lora_acc*100:>9.1f}% {sign}{delta*100:>7.1f}%")

print("-" * 57)
base_total = baseline_summary.accuracy
lora_total = lora_summary.accuracy
delta_total = lora_total - base_total
sign = "+" if delta_total >= 0 else ""
print(f"{'TOTAL':<25} {base_total*100:>9.1f}% {lora_total*100:>9.1f}% {sign}{delta_total*100:>7.1f}%")

## Export Adapter

Package adapter as zip and download. Upload to Kaggle as a private Dataset, then
attach to `submit.ipynb` on Kaggle to produce the final `submission.zip`.

In [ ]:
# -- Package adapter as zip ------------------------------------------
import zipfile, os, glob

SUBMISSION_ZIP = "/content/adapter.zip"

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for filepath in glob.glob(f"{ADAPTER_DIR}/**/*", recursive=True):
        if os.path.isfile(filepath):
            arcname = os.path.relpath(filepath, ADAPTER_DIR)
            zf.write(filepath, arcname)

zip_size = os.path.getsize(SUBMISSION_ZIP) / 1e6
print(f"adapter.zip created: {zip_size:.1f} MB")

# Verify adapter_config.json is present
with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    assert "adapter_config.json" in names, "adapter_config.json missing from zip!"
    print(f"Files in zip: {names}")

# Copy to Drive
import shutil
shutil.copy(SUBMISSION_ZIP, f"{DRIVE_OUT}/adapter_v1.zip")
print(f"Saved to Drive: {DRIVE_OUT}/adapter_v1.zip")

In [ ]:
# -- Download to local machine ---------------------------------------
from google.colab import files
files.download(SUBMISSION_ZIP)
print("Download started.")
print("Next: upload adapter.zip as a Kaggle Dataset, attach to submit.ipynb, run it.")